In [12]:
!wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
!sha256sum data.zip
!unzip -q data.zip

--2026-09-17 12:00:07--  https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/405934815/e712cf72-f851-44e0-9c05-e711624af985?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-17T10%3A33%3A51Z&rscd=attachment%3B+filename%3Ddata.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-17T09%3A33%3A10Z&ske=2026-09-17T10%3A33%3A51Z&sks=b&skv=2018-11-09&sig=WLwisb3qQvuNqumrGGyjiWdiC9cPC9xYUQwbgJgyK4c%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4OTY0MTAwNywibmJmIjo

In [ ]:
!uv venv --clear --python 3.11
!uv pip install -r requirements-cpu.txt

In [24]:
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [25]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

In [26]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [27]:
train_dataset = ImageFolder('data/train', transform=evaluation_transform)
evaluation_dataset = ImageFolder('data/test', transform=evaluation_transform)

In [28]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [29]:
import torch.nn as nn

class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3,32,kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32*99*99, 64)
        self.output = nn.Linear(64,1)

    def forward(self, x):
        x = nn.functional.relu(self.conv(x))
        x = self.pool(x)
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [30]:
model = HairModel()

In [31]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

In [32]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

20073473

In [33]:
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            logits = model(images)
            loss = criterion(logits, labels)
            loss_sum += loss.item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total

In [34]:
model.to(device)

history_baseline = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": [],
}

for epoch in range(10):
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * images.size(0)
        correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
        total += labels.size(0)

    train_loss = loss_sum / total
    train_accuracy = correct / total
    evaluation_loss, evaluation_accuracy = evaluate(model, evaluation_loader, criterion, device)

    history_baseline["train_loss"].append(train_loss)
    history_baseline["train_accuracy"].append(train_accuracy)
    history_baseline["evaluation_loss"].append(evaluation_loss)
    history_baseline["evaluation_accuracy"].append(evaluation_accuracy)

    print(
        f"epoch {epoch + 1}/10  "
        f"train_loss={train_loss:.4f} train_accuracy={train_accuracy:.4f}  "
        f"evaluation_loss={evaluation_loss:.4f} evaluation_accuracy={evaluation_accuracy:.4f}"
    )

epoch 1/10  train_loss=0.6385 train_accuracy=0.6450  evaluation_loss=0.6010 evaluation_accuracy=0.6567
epoch 2/10  train_loss=0.5441 train_accuracy=0.7175  evaluation_loss=0.6003 evaluation_accuracy=0.6517
epoch 3/10  train_loss=0.4955 train_accuracy=0.7550  evaluation_loss=0.6405 evaluation_accuracy=0.6219
epoch 4/10  train_loss=0.4330 train_accuracy=0.7913  evaluation_loss=0.6274 evaluation_accuracy=0.6915
epoch 5/10  train_loss=0.4048 train_accuracy=0.8087  evaluation_loss=0.6293 evaluation_accuracy=0.6617
epoch 6/10  train_loss=0.3953 train_accuracy=0.8063  evaluation_loss=0.6332 evaluation_accuracy=0.7114
epoch 7/10  train_loss=0.3437 train_accuracy=0.8363  evaluation_loss=0.6895 evaluation_accuracy=0.6716
epoch 8/10  train_loss=0.2864 train_accuracy=0.8750  evaluation_loss=0.6526 evaluation_accuracy=0.7065
epoch 9/10  train_loss=0.2224 train_accuracy=0.9125  evaluation_loss=0.6979 evaluation_accuracy=0.7363
epoch 10/10  train_loss=0.1818 train_accuracy=0.9213  evaluation_loss=1.0

In [35]:
import json

with open("history_baseline.json", "w") as f:
    json.dump(history_baseline, f, indent=2)

In [37]:
import pandas as pd

df = pd.read_json('history_baseline.json')

round(df.train_accuracy.median(),2)

np.float64(0.81)

In [39]:
round(df.train_loss.std(ddof=0),3)

np.float64(0.135)

In [40]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [41]:
train_dataset = ImageFolder('data/train', transform=train_transform_augmented)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)

In [43]:
history_augmented = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": [],
}

for epoch in range(10):
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * images.size(0)
        correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
        total += labels.size(0)

    train_loss = loss_sum / total
    train_accuracy = correct / total
    evaluation_loss, evaluation_accuracy = evaluate(model, evaluation_loader, criterion, device)

    history_augmented["train_loss"].append(train_loss)
    history_augmented["train_accuracy"].append(train_accuracy)
    history_augmented["evaluation_loss"].append(evaluation_loss)
    history_augmented["evaluation_accuracy"].append(evaluation_accuracy)

    print(
        f"epoch {epoch + 1}/10  "
        f"train_loss={train_loss:.4f} train_accuracy={train_accuracy:.4f}  "
        f"evaluation_loss={evaluation_loss:.4f} evaluation_accuracy={evaluation_accuracy:.4f}"
    )

epoch 1/10  train_loss=0.5430 train_accuracy=0.7075  evaluation_loss=0.5413 evaluation_accuracy=0.7363
epoch 2/10  train_loss=0.5175 train_accuracy=0.7300  evaluation_loss=0.5599 evaluation_accuracy=0.7413
epoch 3/10  train_loss=0.5065 train_accuracy=0.7300  evaluation_loss=0.5968 evaluation_accuracy=0.6915
epoch 4/10  train_loss=0.4790 train_accuracy=0.7800  evaluation_loss=0.5752 evaluation_accuracy=0.7413
epoch 5/10  train_loss=0.5027 train_accuracy=0.7538  evaluation_loss=0.5528 evaluation_accuracy=0.7363
epoch 6/10  train_loss=0.4926 train_accuracy=0.7625  evaluation_loss=0.6796 evaluation_accuracy=0.6766
epoch 7/10  train_loss=0.4695 train_accuracy=0.7762  evaluation_loss=0.5236 evaluation_accuracy=0.7562
epoch 8/10  train_loss=0.4644 train_accuracy=0.7900  evaluation_loss=0.5112 evaluation_accuracy=0.7512
epoch 9/10  train_loss=0.4516 train_accuracy=0.7900  evaluation_loss=0.5509 evaluation_accuracy=0.7313
epoch 10/10  train_loss=0.4571 train_accuracy=0.7863  evaluation_loss=0.5

In [44]:
with open("history_augmented.json", "w") as f:
    json.dump(history_augmented, f, indent=2)

In [45]:
df = pd.read_json('history_augmented.json')

In [46]:
round(df.evaluation_loss.mean(), 3)

np.float64(0.56)

In [49]:
round(df.evaluation_accuracy[-5:].mean(),2)

np.float64(0.73)